Análisis de Audios de Pacientes de Parkinson 

Este dataset es una versión reducida de la base de datos brindada por el proyecto Parkinson’s Voice Initiative (PVI). Se puede acceder a la misma a través del portal synapse.org mediante la id ‘syn2321745’, la cual está registrada como Patient Voice Analysis (PVA) project.

La base de datos incluye información de una serie de audios brindados para trabajar en la detección temprana de Parkinson. Esta version del dataset incluye la siguiente información de cada paciente: Referencia, Edad, Sexo, Años desde el 1er Sintoma, Diagnóstico en la Escala Hoeh-Yahr, Puntuación UPDRS, Estado del Tratamiento, Duración de la Grabación, Duración del Audio y Duración de Voz Usable. 

In [1]:
#!pip install --force-reinstall pyarrow==10.0.1

In [2]:
import pandas as pd


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/joao/Desktop/HerramientasIA/S4_Practica1/entornoPractica1/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/joao/Desktop/HerramientasIA/S4_Pract

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/Users/joao/Desktop/HerramientasIA/S4_Practica1/entornoPractica1/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/joao/Desktop/HerramientasIA/S4_Pract

AttributeError: _ARRAY_API not found

In [3]:
# se lee el dataset
data = pd.read_excel("Dataset_Parkinson.xlsx")

data

,callref,current_age,sex,years_since_first_symptom,hoehn_yahr,pdrs_score,on_treatment_id,recording_duration,audio_duration,voice_usable_duration
0,23336,63,F,7,1,13,True,28,21.879,13.80
1,63225,76,F,2,1,5,True,30,28.157,28.13
2,99762,51,F,11,2,27,False,28,24.555,20.90
3,110232,67,M,5,1,14,True,22,18.702,13.39
4,128887,58,F,24,3,25,True,23,15.834,13.04
...,...,...,...,...,...,...,...,...,...,...
284,9735253,63,M,16,4,35,True,30,28.430,18.95
285,9764151,70,F,6,1,14,True,23,16.390,16.36
286,9823663,64,F,7,1,14,True,17,14.915,10.33
287,9949188,65,M,3,3,25,True,19,17.177,11.95


In [4]:
# se añade columna de ratio de audio usable por duración total de la grabación
data['usable_voice_ratio'] = (
    data['voice_usable_duration'] / data['recording_duration']
)

In [5]:
# se añaden rangos de edad de los pacientes 

rangos_edad = [40, 50, 60, 70, 80, 100]
labels_rangos_edad = ['40-49', '50-59', '60-69', '70-79', '+80']

data['age_group'] = pd.cut(data['current_age'], bins=rangos_edad, labels=labels_rangos_edad, right=False)


In [6]:
# se añaden rangos de años de convivencia con la enfermedad 

rangos_convivencia = [0, 8, 15, 22, 29, 36, 43, 50]
labels_rangos_convivencia = ['0-7', '8-14', '15-21', '22-28', '29-35', '36-42', '43-49']

data['years_group_with_Parkinson'] = pd.cut(
    data['years_since_first_symptom'],
    bins=rangos_convivencia,
    labels=labels_rangos_convivencia,
    right=False
)


In [7]:
# se añade una evaluación del nivel de parkinson en función de
# la escala Hoehn-Yahr y la puntuación PDRS
def parkinson_level(row):
    hoehn_yahr = row['hoehn_yahr']
    pdrs = row['pdrs_score']
    
    if hoehn_yahr <= 2 and pdrs <= 20:
        return 'Early'
    elif (2 < hoehn_yahr <= 3) or (20 < pdrs <= 40):
        return 'Medium'
    else:
        return 'Severe'

data['parkinson_level'] = data.apply(parkinson_level, axis=1)



In [8]:
data.style.format({
    'usable_voice_ratio': '{:.4%}'
})

,callref,current_age,sex,years_since_first_symptom,hoehn_yahr,pdrs_score,on_treatment_id,recording_duration,audio_duration,voice_usable_duration,usable_voice_ratio,age_group,years_group_with_Parkinson,parkinson_level
0,23336,63,F,7,1,13,True,28,21.879000,13.800000,49.2857%,60-69,0-7,Early
1,63225,76,F,2,1,5,True,30,28.157000,28.130000,93.7667%,70-79,0-7,Early
2,99762,51,F,11,2,27,False,28,24.555000,20.900000,74.6429%,50-59,8-14,Medium
3,110232,67,M,5,1,14,True,22,18.702000,13.390000,60.8636%,60-69,0-7,Early
4,128887,58,F,24,3,25,True,23,15.834000,13.040000,56.6957%,50-59,22-28,Medium
5,149714,52,F,2,3,26,True,14,7.128000,5.320000,38.0000%,50-59,0-7,Medium
6,194489,61,F,5,1,10,True,18,12.342000,11.010000,61.1667%,60-69,0-7,Early
7,208635,58,F,17,1,26,True,30,28.993000,28.970000,96.5667%,50-59,15-21,Medium
8,294505,68,F,11,5,50,True,20,17.427000,12.320000,61.6000%,60-69,8-14,Severe
9,296853,62,M,9,3,50,True,17,15.706000,10.760000,63.2941%,60-69,8-14,Medium


In [9]:
import pygwalker as pyg

In [11]:
import ipywidgets as widgets
widgets.IntSlider()

IntSlider(value=0)

In [ ]:
pyg.walk(data)

Box(children=(HTML(value='\n<div id="ifr-pyg-000650f4bf30c5e0zKkwjVJgByaHqAvN" style="height: auto">\n    <hea…

In [14]:
!pip install --force-reinstall \
  "ipywidgets>=8.1" \
  "jupyterlab_widgets>=3.0"

  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
  Using cached comm-0.2.3-py3-none-any.whl.metadata (3.7 kB)
  Using cached ipython-8.39.0-py3-none-any.whl.metadata (5.1 kB)
  Using cached traitlets-5.14.3-py3-none-any.whl.metadata (10 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached decorator-5.2.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached exceptiongroup-1.3.1-py3-none-any.whl.metadata (6.7 kB)
  Using cached jedi-0.20.0-py2.py3-none-any.whl.metadata (23 kB)
  Using cached matplotlib_inline-0.2.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached pexpect-4.9.0-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached prompt_toolkit-3.0.52-py3-none-any.whl.metadata (6.4 kB)
  Using cached pygments-2.20.0-py3-none-any.whl.metadata (2.5 kB)
  Using cached stack_data-0.6.3-py3-none-any.whl.metadata (18 kB)
  Using cached typing_extensions-4.15.0-py